In [2]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from cdt.data import AcyclicGraphGenerator
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
from causallearn.graph.GeneralGraph import GeneralGraph
from collections import Counter
from pathlib import Path
import os

Detecting 1 CUDA device(s).


In [30]:
n = 1000
p = 10
X_1 = np.random.standard_normal(size=(n,1))
data = np.zeros(shape=(n,p))
data[:,0] = X_1.squeeze()

for i in range(1,p):
    j = np.random.randint(0,high=i-1) if i > 1 else 0
    a = np.random.random(size=1)
    b = np.random.random(size=1)
    data[:,i] = (a * data[:,j] + b) + np.random.standard_normal(size=(n,1)).T

In [27]:
data = np.random.standard_normal(size=(1000,10))
cgraph = pc(data)
cgraph.G.graph

  0%|          | 0/10 [00:00<?, ?it/s]

array([[ 0,  0, -1,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [-1,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0, -1,  0,  0,  0,  0,  0],
       [ 0,  0,  0, -1,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0, -1,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0, -1,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0]])

In [31]:
from causallearn.search.ScoreBased.ExactSearch import bic_exact_search

dag = bic_exact_search(data)

In [32]:
dag

(array([[0., 1., 1., 1., 1., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 1., 0.],
        [0., 1., 0., 0., 0., 1., 1., 0., 1., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 1., 0., 0., 0., 1., 0., 1.],
        [0., 1., 0., 0., 1., 1., 0., 1., 1., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 1., 0.]]),
 {'n_parent_graphs_entries': 997,
  'while_iter': 789,
  'for_iter': 4198,
  'n_closed': 789,
  'max_n_opened': 360})

In [ ]:
def triangular_reordering(
    matrix: np.ndarray,
    *,
    tolerance: float = 0.0,
) -> tuple[bool, np.ndarray | None]:
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError("Matrix must be square.")

    # A[i, j] represents an edge from node i to node j.
    adjacency = np.abs(matrix) > tolerance
    graph = nx.from_numpy_array(
        adjacency,
        create_using=nx.DiGraph,
    )

    try:
        order = np.array(list(nx.topological_sort(graph)), dtype=int)
    except nx.NetworkXUnfeasible:
        return False, None

    return True, order

In [34]:
triangular_reordering(dag[0])

(True, array([0, 2, 6, 5, 3, 7, 9, 1, 4, 8]))